# Notebook 11 — API REST con FastAPI + MongoDB + ngrok
### Investing AI · iDeSo · UNMSM

**Objetivo:** Levantar una API REST que **LEE datos pre-calculados de MongoDB** (no recalcula) y los expone vía HTTP mediante ngrok.

**Prerequisitos:**
- `MONGO_URI` y `NGROK_AUTHTOKEN` configurados en los Secrets de Colab (`🔑`).
- Notebook 1 (Ingesta) y Notebook 2 (SVC) ejecutados y con datos en MongoDB.
- Notebook 9 (Optimizador de Portafolio) y Notebook 10 (Backtesting) ejecutados para las colecciones `estrategias` y `backtests`.

**Endpoints implementados:**

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| GET | `/api/salud` | Estado del servidor y conexión a MongoDB |
| GET | `/api/mercado/{ticker}` | Datos OHLCV + indicadores técnicos |
| GET | `/api/svc/{ticker}` | Señal BUY/SELL y métricas del clasificador SVC |
| GET | `/api/rnns/{ticker}` | *(Extra)* Señales y métricas de clasificadores RNN |
| GET | `/api/lstm/{ticker}` | *(Extra)* Pronóstico de precio futuro (regresor LSTM) |
| GET | `/api/noticias` | *(Extra)* Feed de noticias con sentimiento NLP (VADER), filtrable por `fuente` y `sentimiento` |
| GET | `/api/estrategias` | *(Extra)* Asignación óptima de portafolio (Markowitz), filtrable por `perfil_riesgo`, `horizonte` y `capital` |
| GET | `/api/backtests` | *(Extra)* Backtest de portafolio diversificado — alimenta las pestañas Backtesting y Portafolio, filtrable por `modelo`, `perfil_riesgo` y `capital` |\n| POST/GET | `/api/ordenes` | *(Extra)* Libro de órdenes manual del módulo Broker (registrar y listar órdenes, con validación de fondos/posición) |
| GET | `/api/cuenta` | *(Extra)* Poder adquisitivo y posiciones abiertas del Broker, derivados del historial de órdenes |

**Pipeline completo:**
```
yfinance → MongoDB (NB1) → SVC (NB2) → MongoDB → FastAPI (NB3) → Frontend HTML
NB9 (Estrategias Markowitz) + NB10 (Backtesting) → MongoDB → FastAPI (NB3) → Frontend HTML
```


## Paso 1 — Instalación de librerías

In [95]:
# ============================================================
# PASO 1: Instalar todas las dependencias necesarias
# ============================================================
!pip install fastapi uvicorn pyngrok nest-asyncio 'pymongo[srv]' --quiet

print("✓ Librerías instaladas correctamente")
print("  fastapi    — framework web para la API REST")
print("  uvicorn    — servidor ASGI para ejecutar FastAPI")
print("  pyngrok    — túnel ngrok para exponer la API a Internet")
print("  nest-asyncio — permite asyncio dentro de Jupyter/Colab")
print("  pymongo    — cliente oficial de MongoDB para Python")


✓ Librerías instaladas correctamente
  fastapi    — framework web para la API REST
  uvicorn    — servidor ASGI para ejecutar FastAPI
  pyngrok    — túnel ngrok para exponer la API a Internet
  nest-asyncio — permite asyncio dentro de Jupyter/Colab
  pymongo    — cliente oficial de MongoDB para Python


## Paso 2 — Importaciones y configuración global

In [96]:
# ============================================================
# PASO 2: Importaciones y configuración global
# ============================================================
import json
import time
import threading
import warnings
from datetime import datetime
from typing import Any, Dict, List, Optional

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, ServerSelectionTimeoutError
from pyngrok import conf, ngrok

# Silenciar advertencias menores
warnings.filterwarnings('ignore')

# Aplicar nest_asyncio para que uvicorn funcione dentro de Colab/Jupyter
nest_asyncio.apply()

print("✓ Importaciones completadas")


✓ Importaciones completadas


## Paso 3 — Configurar credenciales (Secrets de Colab)

In [97]:
# ============================================================
# PASO 3: Leer MONGO_URI y NGROK_AUTHTOKEN desde los Secrets
# de Colab (ícono de llave 🔑 en el panel izquierdo).
#
# IMPORTANTE: nunca pegues credenciales directamente en el
# código; usa siempre los Secrets para proteger tus datos.
# ============================================================
from google.colab import userdata

# ── ngrok: token de autenticación ────────────────────────────
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
conf.get_default().auth_token = NGROK_TOKEN  # pyngrok 8.x
print("✓ ngrok configurado")

# ── MongoDB: URI de conexión ──────────────────────────────────
MONGO_URI = userdata.get('MONGO_URI')

# ── Nombre de la base de datos ────────────────────────────────
DB_NAME  = 'spbi'

# ── Colecciones que usa la API (solo lectura) ─────────────────
COL_PRECIOS     = 'precios_ohlcv'       # Notebook 1
COL_PREDICCIONES = 'predicciones'       # Notebook 2
COL_METRICAS    = 'metricas_modelos'    # Notebook 2
COL_NOTICIAS    = 'noticias'            # Notebook 8 (NLP / VADER)
COL_ESTRATEGIAS = 'estrategias'         # Notebook 9  (Optimizador Markowitz)
COL_BACKTESTS   = 'backtests'           # Notebook 10 (Backtesting de Portafolio)
COL_ORDENES     = 'ordenes_simuladas'   # Broker (libro de órdenes manual del usuario)
# COL_RNN ya no se usa: el LSTM comparte 'predicciones' y 'metricas_modelos' con el SVC (filtrando modelo='LSTM')

print("✓ Variables de configuración definidas")
print(f"  Base de datos : {DB_NAME}")
print(f"  Colecciones   : {COL_PRECIOS}, {COL_PREDICCIONES}, {COL_METRICAS}, {COL_NOTICIAS}, {COL_ESTRATEGIAS}, {COL_BACKTESTS}")


✓ ngrok configurado
✓ Variables de configuración definidas
  Base de datos : spbi
  Colecciones   : precios_ohlcv, predicciones, metricas_modelos, noticias, estrategias, backtests


## Paso 4 — Conexión a MongoDB Atlas

In [98]:
# ============================================================
# PASO 4: Conectar a MongoDB Atlas y verificar la conexión
# ============================================================

def crear_cliente_mongo(uri: str, timeout_ms: int = 5000) -> MongoClient:
    """
    Crea un cliente MongoDB con timeout configurable.
    Lanza ConnectionError si no se puede conectar.
    """
    cliente = MongoClient(uri, serverSelectionTimeoutMS=timeout_ms)
    # Ping rápido para verificar que la conexión es válida
    cliente.admin.command('ping')
    return cliente

try:
    cliente_mongo = crear_cliente_mongo(MONGO_URI)
    db = cliente_mongo[DB_NAME]

    # Contar documentos en cada colección para verificar que hay datos
    n_precios  = db[COL_PRECIOS].count_documents({})
    n_pred     = db[COL_PREDICCIONES].count_documents({})
    n_met      = db[COL_METRICAS].count_documents({})
    n_rnn      = db[COL_PREDICCIONES].count_documents({'modelo': 'LSTM'})
    n_noticias = db[COL_NOTICIAS].count_documents({})
    n_estrategias = db[COL_ESTRATEGIAS].count_documents({})
    n_backtests   = db[COL_BACKTESTS].count_documents({})

    print("✓ Conexión a MongoDB Atlas exitosa")
    print(f"  Base de datos       : {DB_NAME}")
    print(f"  {COL_PRECIOS:<22}: {n_precios:,} documentos")
    print(f"  {COL_PREDICCIONES:<22}: {n_pred:,} documentos")
    print(f"  {COL_METRICAS:<22}: {n_met:,} documentos")
    print(f"  predicciones (LSTM)  : {n_rnn:,} documentos")
    print(f"  {COL_NOTICIAS:<22}: {n_noticias:,} documentos")
    print(f"  {COL_ESTRATEGIAS:<22}: {n_estrategias:,} documentos")
    print(f"  {COL_BACKTESTS:<22}: {n_backtests:,} documentos")

    if n_precios == 0:
        print("\n⚠  AVISO: 'precios_ohlcv' vacío — ejecuta el Notebook 1 primero")
    if n_pred == 0:
        print("⚠  AVISO: 'predicciones' vacío — ejecuta el Notebook 2 primero")
    if n_noticias == 0:
        print("⚠  AVISO: 'noticias' vacío — ejecuta el Notebook 8 (NLP) para el módulo M5")
    if n_estrategias == 0:
        print("⚠  AVISO: 'estrategias' vacío — ejecuta el Notebook 9 (Optimizador Markowitz) para la pestaña Estrategias")
    if n_backtests == 0:
        print("⚠  AVISO: 'backtests' vacío — ejecuta el Notebook 10 (Backtesting) para las pestañas Portafolio y Backtesting")

except (ConnectionFailure, ServerSelectionTimeoutError) as e:
    raise RuntimeError(
        f"❌ No se pudo conectar a MongoDB Atlas.\n"
        f"   Verifica que MONGO_URI sea correcto y que el IP 0.0.0.0/0 "
        f"esté en la lista de acceso de red en Atlas.\n"
        f"   Error: {e}"
    )


✓ Conexión a MongoDB Atlas exitosa
  Base de datos       : spbi
  precios_ohlcv         : 1,253 documentos
  predicciones          : 29 documentos
  metricas_modelos      : 29 documentos
  predicciones (LSTM)  : 5 documentos
  noticias              : 46 documentos
  estrategias           : 12 documentos
  backtests             : 12 documentos


## Paso 5 — Crear la aplicación FastAPI

In [99]:
# ============================================================
# PASO 5: Crear la aplicación FastAPI con CORS habilitado
#
# CORS (Cross-Origin Resource Sharing) es necesario para que
# el frontend HTML (desplegado en GitHub Pages u otro dominio)
# pueda hacer fetch() a esta API sin ser bloqueado por el
# navegador.
# ============================================================

app = FastAPI(
    title="Ernesto Investing AI — API REST",
    description=(
        "API que expone datos de mercado y predicciones del "
        "clasificador SVC para los 5 activos mineros del proyecto. "
        "Lee datos pre-calculados de MongoDB (no recalcula en tiempo real)."
    ),
    version="1.0.0",
    docs_url="/docs",      # Swagger UI
    redoc_url="/redoc",    # ReDoc (documentación alternativa)
)

# Habilitar CORS para que cualquier origen pueda consumir la API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # Cualquier origen (frontend en GitHub Pages, etc.)
    allow_credentials=True,
    allow_methods=["*"],          # GET, POST, OPTIONS, etc.
    allow_headers=["*"],          # Incluyendo ngrok-skip-browser-warning
)

# ── Tickers soportados por el sistema ────────────────────────
TICKERS_VALIDOS = {'FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP'}

def validar_ticker(ticker: str) -> str:
    """
    Normaliza el ticker a mayúsculas y verifica que esté
    en la lista de activos soportados.
    """
    t = ticker.upper()
    if t not in TICKERS_VALIDOS:
        raise HTTPException(
            status_code=404,
            detail={
                "error": f"Ticker '{t}' no está en el sistema.",
                "tickers_disponibles": sorted(TICKERS_VALIDOS),
                "sugerencia": "Verifica el símbolo del ticker (distingue mayúsculas/minúsculas en Yahoo Finance)."
            }
        )
    return t

print("✓ Aplicación FastAPI creada")
print(f"  Tickers soportados: {sorted(TICKERS_VALIDOS)}")


✓ Aplicación FastAPI creada
  Tickers soportados: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']


## Paso 6 — Endpoint `/api/salud`

In [100]:
# ============================================================
# PASO 6: Endpoint de salud del sistema
#
# Verifica que:
# 1. El servidor FastAPI está levantado y respondiendo.
# 2. La conexión a MongoDB sigue activa.
# 3. Las colecciones tienen datos (Notebooks 1 y 2 ejecutados).
# ============================================================

@app.get(
    "/api/salud",
    summary="Estado del servidor",
    tags=["Sistema"],
    response_description="Estado operativo del servidor y la base de datos"
)
def salud():
    """
    Verifica el estado del servidor y la conexión a MongoDB.

    Retorna:
    - **estado**: 'operativo' o 'degradado'
    - **base_datos**: 'conectada' o 'error'
    - Conteo de documentos en cada colección
    """
    try:
        # Ping a MongoDB para verificar que la conexión sigue viva
        cliente_mongo.admin.command("ping")

        n_precios = db[COL_PRECIOS].count_documents({})
        n_pred    = db[COL_PREDICCIONES].count_documents({})
        n_met     = db[COL_METRICAS].count_documents({})
        n_estr    = db[COL_ESTRATEGIAS].count_documents({})
        n_back    = db[COL_BACKTESTS].count_documents({})

        # Si faltan datos, el estado es 'degradado' pero sigue funcionando
        datos_completos = (n_precios > 0 and n_pred > 0 and n_met > 0 and n_estr > 0 and n_back > 0)

        return {
            "estado"               : "operativo" if datos_completos else "degradado",
            "base_datos"           : "conectada",
            "colecciones": {
                COL_PRECIOS      : n_precios,
                COL_PREDICCIONES : n_pred,
                COL_METRICAS     : n_met,
                COL_ESTRATEGIAS  : n_estr,
                COL_BACKTESTS    : n_back,
            },
            "tickers_disponibles"  : sorted(TICKERS_VALIDOS),
            "timestamp"            : datetime.now().isoformat(),
            "version_api"          : "1.0.0",
            "advertencia"          : None if datos_completos else
                                     "Una o más colecciones están vacías. Ejecuta los Notebooks 1, 2, 9 y 10."
        }

    except Exception as e:
        # Si MongoDB no responde, retorna 503 (servicio no disponible)
        raise HTTPException(
            status_code=503,
            detail={
                "estado"     : "error",
                "base_datos" : "desconectada",
                "error"      : str(e),
                "timestamp"  : datetime.now().isoformat()
            }
        )

print("✓ Endpoint /api/salud registrado")


✓ Endpoint /api/salud registrado


## Paso 7 — Endpoint `/api/mercado/{ticker}`

In [101]:
# ============================================================
# PASO 7: Endpoint de datos de mercado
#
# Lee de la colección 'precios_ohlcv' los datos OHLCV e
# indicadores técnicos calculados por el Notebook 1.
#
# El parámetro 'dias' permite solicitar los últimos N días
# (por defecto: todos los disponibles).
# ============================================================

@app.get(
    "/api/mercado/{ticker}",
    summary="Datos OHLCV e indicadores técnicos",
    tags=["Mercado"],
    response_description="Serie histórica de precios con SMA, EMA y RSI"
)
def mercado(
    ticker: str,
    dias: Optional[int] = Query(
        default=None,
        ge=1,
        le=1500,
        description="Últimos N días de datos. Sin valor → devuelve todo el histórico disponible."
    )
):
    """
    Retorna datos OHLCV + indicadores técnicos de un ticker.

    **Indicadores disponibles:** SMA-20, SMA-50, EMA-12, EMA-26, RSI-14, MACD, Bandas de Bollinger.

    **Parámetro opcional:** `?dias=90` para los últimos 90 días.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)

    # Campos a excluir de la respuesta de MongoDB
    proyeccion = {"_id": 0, "created_at": 0}

    # Recuperar documentos ordenados cronológicamente
    cursor = db[COL_PRECIOS].find({"ticker": t}, proyeccion).sort("fecha", 1)
    datos  = list(cursor)

    if not datos:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay datos de mercado para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook 1 (Ingesta) para poblar la colección precios_ohlcv.",
                "ticker"     : t
            }
        )

    # Aplicar filtro de días si se especificó
    if dias is not None:
        datos = datos[-dias:]

    # Calcular rango de fechas disponibles
    fecha_inicio = datos[0].get("fecha",  "—")
    fecha_fin    = datos[-1].get("fecha", "—")
    ultimo_precio = datos[-1].get("close", None)

    return {
        "ticker"           : t,
        "empresa"          : datos[-1].get("company_name", t),
        "registros"        : len(datos),
        "fecha_inicio"     : fecha_inicio,
        "fecha_fin"        : fecha_fin,
        "ultimo_precio"    : ultimo_precio,
        "indicadores"      : ["sma_20", "sma_50", "ema_12", "ema_26", "rsi_14",
                               "macd_line", "macd_signal", "macd_histogram",
                               "bb_upper", "bb_middle", "bb_lower"],
        "datos"            : datos
    }

print("✓ Endpoint /api/mercado/{ticker} registrado")


✓ Endpoint /api/mercado/{ticker} registrado


## Paso 8 — Endpoint `/api/svc/{ticker}`

In [102]:
# ============================================================
# PASO 8: Endpoint del clasificador SVC
#
# Lee de las colecciones 'predicciones' y 'metricas_modelos'
# los resultados generados por el Notebook 2 (SVC).
#
# La API NO recalcula: solo sirve lo que ya está en MongoDB.
# Esto garantiza respuestas en milisegundos.
# ============================================================

@app.get(
    "/api/svc/{ticker}",
    summary="Señal BUY/SELL y métricas del clasificador SVC",
    tags=["Clasificador SVC"],
    response_description="Predicción actual, métricas del modelo y matriz de confusión"
)
def svc(ticker: str):
    """
    Retorna la señal de trading actual (BUY/SELL) y las métricas
    de evaluación del clasificador SVC para un ticker.

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por el Notebook 2.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción actual ──────────────────────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": "SVC"},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay predicción SVC para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook 2 (SVC) para generar predicciones.",
                "ticker"     : t
            }
        )

    # ── 2. Métricas del modelo ────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": "SVC"},
        {**proyeccion, "fecha_entrenamiento": 0}
    )

    return {
        "ticker"            : t,
        "modelo"            : "SVC",
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "tipo_features"     : prediccion.get("tipo_features"),
        "prediccion": {
            "senal"      : prediccion.get("senal"),
            "confianza"  : prediccion.get("confianza"),
        },
        "metricas": {
            "accuracy"          : metricas.get("accuracy")   if metricas else None,
            "precision"         : metricas.get("precision")  if metricas else None,
            "recall"            : metricas.get("recall")     if metricas else None,
            "f1"                : metricas.get("f1")         if metricas else None,
            "matriz_confusion"  : metricas.get("matriz_confusion") if metricas else None,
            "mejor_kernel"      : metricas.get("mejor_kernel")     if metricas else None,
            "mejor_C"           : metricas.get("mejor_C")          if metricas else None,
            "mejor_gamma"       : metricas.get("mejor_gamma")      if metricas else None,
        },
        "historico_senales" : prediccion.get("historico_senales", [])
    }

print("✓ Endpoint /api/svc/{ticker} registrado (campos alineados con el Notebook 2 real)")


✓ Endpoint /api/svc/{ticker} registrado (campos alineados con el Notebook 2 real)


## Paso 9 — Endpoint `/api/rnns/{ticker}` *(Extra — clasificador LSTM)*

In [103]:
# ============================================================
# PASO 9 (EXTRA): Endpoint para los clasificadores RNN
# (LSTM / BiLSTM / GRU / SimpleRNN)
#
# Lee de las MISMAS colecciones que el SVC ('predicciones' y
# 'metricas_modelos'), filtrando por el campo 'modelo' que el
# usuario elige con el parámetro de query ?modelo=. Se mantiene
# un solo patrón de colecciones para todas las arquitecturas,
# en vez de una colección aparte por modelo.
#
# Si no hay documentos con el modelo pedido para el ticker,
# retorna 404 con un mensaje claro en vez de crashear el servidor.
# ============================================================

MODELOS_RNN_VALIDOS = {'LSTM', 'BiLSTM', 'GRU', 'SimpleRNN'}

@app.get(
    "/api/rnns/{ticker}",
    summary="Señal BUY/SELL y métricas de un clasificador RNN (LSTM/BiLSTM/GRU/SimpleRNN)",
    tags=["Clasificadores RNN"],
    response_description="Predicción actual, métricas del modelo, matriz de confusión e historial de épocas"
)
def rnns(ticker: str, modelo: str = "LSTM"):
    """
    Retorna la señal de trading actual (BUY/SELL) y las métricas
    de evaluación de la arquitectura RNN elegida para un ticker.

    **Parámetro `modelo`:** uno de `LSTM`, `BiLSTM`, `GRU`, `SimpleRNN`
    (por defecto `LSTM`). Corresponde al selector de arquitectura del
    frontend (Módulo M3).

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por los Notebooks 3 (LSTM), 4 (BiLSTM), 5 (GRU) y 6 (SimpleRNN),
    filtrando por el campo `modelo`.

    Este endpoint es opcional: si el notebook correspondiente no ha sido
    ejecutado para este ticker/modelo, retorna 404 con un mensaje descriptivo.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    m = modelo.strip()

    if m not in MODELOS_RNN_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error"             : f"Modelo '{m}' no reconocido.",
                "modelos_validos"   : sorted(MODELOS_RNN_VALIDOS),
                "sugerencia"        : "Usa ?modelo=LSTM, ?modelo=BiLSTM, ?modelo=GRU o ?modelo=SimpleRNN."
            }
        )

    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción actual ──────────────────────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": m},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay predicción {m} para '{t}'.",
                "sugerencia" : f"Ejecuta el notebook del clasificador {m} para este ticker.",
                "ticker"     : t,
                "modelo"     : m
            }
        )

    # ── 2. Métricas del modelo ────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": m},
        {**proyeccion, "fecha_entrenamiento": 0}
    )

    return {
        "ticker"            : t,
        "modelo"            : m,
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "n_steps"           : prediccion.get("n_steps"),
        "prediccion": {
            "senal"      : prediccion.get("senal"),
            "confianza"  : prediccion.get("confianza"),
        },
        "metricas": {
            "accuracy"          : metricas.get("accuracy")   if metricas else None,
            "precision"         : metricas.get("precision")  if metricas else None,
            "recall"            : metricas.get("recall")     if metricas else None,
            "f1"                : metricas.get("f1")         if metricas else None,
            "epocas_entrenadas" : metricas.get("epocas_entrenadas") if metricas else None,
            "matriz_confusion"  : metricas.get("matriz_confusion")  if metricas else None,
            "historial_epocas"  : metricas.get("historial_epocas")  if metricas else None,
        },
        "historico_senales" : prediccion.get("historico_senales", [])
    }

print("✓ Endpoint /api/rnns/{ticker}?modelo=LSTM|BiLSTM|GRU|SimpleRNN registrado")


✓ Endpoint /api/rnns/{ticker}?modelo=LSTM|BiLSTM|GRU|SimpleRNN registrado


## Paso 10 — Endpoint `/api/lstm/{ticker}` *(Extra — regresor de precios)*

In [104]:
# ============================================================
# PASO 10 (EXTRA): Endpoint del regresor LSTM (precio futuro)
#
# Lee de las MISMAS colecciones que los clasificadores
# ('predicciones' y 'metricas_modelos'), filtrando por
# modelo='LSTM_REG'. Este endpoint es de REGRESIÓN, no de
# clasificación: devuelve un precio continuo proyectado a
# 7/14/30/60 días, con banda de confianza, en vez de una
# señal BUY/SELL.
#
# Traduce el esquema interno de Mongo (proyecciones_horizonte
# con claves '7d'/'14d'/'30d'/'60d') al esquema que consume
# el frontend (prediccion_futura con claves '7_dias'/etc.).
# ============================================================

HORIZONTES_DISPONIBLES = [7, 14, 30, 60]

@app.get(
    "/api/lstm/{ticker}",
    summary="Pronóstico de precio futuro (regresor LSTM)",
    tags=["Regresor LSTM"],
    response_description="Precio actual, histórico real vs. predicho, y proyección a futuro con bandas de confianza"
)
def lstm_regresor(ticker: str, horizonte: int = 60):
    """
    Retorna el pronóstico de precio futuro del regresor LSTM para un ticker.

    **Fuente:** Colecciones `predicciones` y `metricas_modelos` de MongoDB,
    pobladas por el Notebook del Regresor LSTM (`modelo='LSTM_REG'`).

    El parámetro `horizonte` se acepta por compatibilidad con el frontend
    (slider de 7/14/30/60 días), pero la API siempre devuelve las 4
    proyecciones disponibles en `prediccion_futura`; el frontend elige
    cuál mostrar según el horizonte seleccionado.

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    t = validar_ticker(ticker)
    proyeccion = {"_id": 0, "created_at": 0}

    # ── 1. Predicción + proyecciones a futuro ─────────────────
    prediccion = db[COL_PREDICCIONES].find_one(
        {"ticker": t, "modelo": "LSTM_REG"},
        proyeccion
    )
    if not prediccion:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay pronóstico LSTM_REG para '{t}'.",
                "sugerencia" : "Ejecuta el Notebook del Regresor LSTM para este ticker.",
                "ticker"     : t
            }
        )

    # ── 2. Métricas del modelo ─────────────────────────────────
    metricas = db[COL_METRICAS].find_one(
        {"ticker": t, "modelo": "LSTM_REG"},
        {**proyeccion, "fecha_entrenamiento": 0}
    ) or {}

    # ── 3. Traducir 'proyecciones_horizonte' (esquema interno) a
    #        'prediccion_futura' (esquema esperado por el frontend) ──
    horizonte_interno = prediccion.get("proyecciones_horizonte", {})
    prediccion_futura = {}
    for h in HORIZONTES_DISPONIBLES:
        datos_h = horizonte_interno.get(f"{h}d")
        if datos_h:
            prediccion_futura[f"{h}_dias"] = {
                "precio"    : datos_h.get("precio_estimado"),
                "banda_sup" : datos_h.get("banda_superior"),
                "banda_inf" : datos_h.get("banda_inferior"),
            }

    return {
        "ticker"            : t,
        "modelo"            : "LSTM_REG",
        "fecha_prediccion"  : prediccion.get("fecha_prediccion", "—"),
        "n_steps"           : prediccion.get("n_steps"),
        "precio_actual_usd" : prediccion.get("ultimo_precio"),
        "metricas": {
            "rmse_usd"           : metricas.get("rmse_usd"),
            "rmse_pct"           : metricas.get("rmse_pct"),
            "mae_usd"            : metricas.get("mae_usd"),
            "r2"                 : metricas.get("r2"),
            "rmse_arima_baseline": metricas.get("rmse_arima_baseline"),
            "epocas_entrenadas"  : metricas.get("epocas_entrenadas"),
            "historial_epocas"   : metricas.get("historial_epocas"),
        },
        "historico_precios" : prediccion.get("historico_predicciones", []),
        "serie_diaria_60d"  : prediccion.get("serie_diaria_60d", []),
        "prediccion_futura" : prediccion_futura
    }

print("✓ Endpoint /api/lstm/{ticker} registrado (regresor de precios, modelo='LSTM_REG')")


✓ Endpoint /api/lstm/{ticker} registrado (regresor de precios, modelo='LSTM_REG')


## Paso 11 — Endpoint `/api/noticias` *(Extra — sentimiento NLP)*


In [106]:
# ============================================================
# PASO 11 (EXTRA): Endpoint del feed de noticias con sentimiento NLP
#
# Lee la colección 'noticias' poblada por el Notebook 8 (VADER).
# Replica EXACTAMENTE los dos filtros que expone el selector del
# Módulo M5 del frontend:
#   - fuente:      'all' | 'bloomberg' | 'cnbc' | 'reuters'
#   - sentimiento: 'all' | 'bullish' | 'bearish' | 'neutral'
#
# Los <select> del HTML mandan los valores en minúsculas, pero
# MongoDB guarda 'fuente' con mayúscula inicial (Bloomberg, CNBC,
# Reuters, MarketWatch) y 'sentimiento' siempre en mayúsculas
# (BULLISH/BEARISH/NEUTRAL). Por eso la comparación de 'fuente' es
# case-insensitive (regex) y 'sentimiento' se normaliza a upper().
# ============================================================

COL_NOTICIAS_TAG = "NLP"

FUENTES_VALIDAS = {'bloomberg', 'cnbc', 'reuters', 'marketwatch'}
SENTIMIENTOS_VALIDOS = {'bullish', 'bearish', 'neutral'}

@app.get(
    "/api/noticias",
    summary="Feed de noticias con sentimiento NLP (VADER)",
    tags=["NLP"],
    response_description="Noticias filtradas por fuente y sentimiento, con sentimiento consolidado del conjunto"
)
def noticias(
    fuente: str = Query(
        default="all",
        description="'all', 'bloomberg', 'cnbc' o 'reuters' (case-insensitive, igual que el <select> del frontend)"
    ),
    sentimiento: str = Query(
        default="all",
        description="'all', 'bullish', 'bearish' o 'neutral'"
    ),
    ticker: Optional[str] = Query(
        default=None,
        description="Opcional: filtra por un ticker específico (el Módulo M5 no lo usa hoy, pero queda disponible)"
    ),
    limite: int = Query(
        default=50, ge=1, le=200,
        description="Máximo de noticias a devolver, ordenadas por fecha de publicación descendente"
    )
):
    """
    Retorna el feed de noticias procesado por el Notebook 8 (VADER),
    con los mismos filtros que usan los selectores del Módulo M5:

    - **fuente**: `all` | `bloomberg` | `cnbc` | `reuters`
    - **sentimiento**: `all` | `bullish` | `bearish` | `neutral`

    **Fuente de datos:** colección `noticias` de MongoDB, poblada por el
    Notebook 8 (VADER sobre titulares de yfinance + respaldo simulado).

    **Importante:** si el filtro no tiene coincidencias (ej. no hay noticias
    'Reuters' + 'Bearish'), la respuesta es `200 OK` con `"noticias": []` y
    `"mensaje"` explicando que no hay resultados — NO se lanza un 404. Un
    404 solo ocurre si la colección `noticias` está completamente vacía
    (el Notebook 8 nunca se ejecutó).

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    query: Dict[str, Any] = {}

    # ── Filtro de ticker (opcional) ─────────────────────────────
    if ticker:
        query["ticker"] = validar_ticker(ticker)

    # ── Filtro de fuente (case-insensitive) ─────────────────────
    f = fuente.strip().lower()
    if f != "all":
        if f not in FUENTES_VALIDAS:
            raise HTTPException(
                status_code=400,
                detail={
                    "error": f"Fuente '{fuente}' no reconocida.",
                    "fuentes_validas": sorted(FUENTES_VALIDAS) + ["all"]
                }
            )
        query["fuente"] = {"$regex": f"^{f}$", "$options": "i"}

    # ── Filtro de sentimiento ────────────────────────────────────
    s = sentimiento.strip().lower()
    if s != "all":
        if s not in SENTIMIENTOS_VALIDOS:
            raise HTTPException(
                status_code=400,
                detail={
                    "error": f"Sentimiento '{sentimiento}' no reconocido.",
                    "sentimientos_validos": sorted(SENTIMIENTOS_VALIDOS) + ["all"]
                }
            )
        query["sentimiento"] = s.upper()  # BULLISH / BEARISH / NEUTRAL

    # ── ¿La colección está vacía por completo? (Notebook 8 nunca ejecutado) ──
    # Esto SÍ es un error real de configuración, y se distingue de "el filtro
    # elegido no tiene coincidencias", que es un resultado válido, no un error.
    total_coleccion = db[COL_NOTICIAS].estimated_document_count()
    if total_coleccion == 0:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : "La colección 'noticias' está vacía (no hay ningún documento).",
                "sugerencia" : "Ejecuta el Notebook 8 (NLP) para poblar la colección 'noticias'."
            }
        )

    proyeccion = {"_id": 0, "created_at": 0}
    cursor = db[COL_NOTICIAS].find(query, proyeccion).sort("fecha_publicacion", -1).limit(limite)
    docs = list(cursor)

    # ── Sentimiento consolidado del conjunto filtrado ────────────
    # Si el filtro no tiene coincidencias, docs=[] y esto se queda en None;
    # NO se lanza 404: un filtro sin resultados es una respuesta 200 válida,
    # y así debe interpretarlo el frontend (mostrar "sin noticias", no
    # "sin conexión").
    compounds = [d.get("compound", 0.0) for d in docs]
    compound_promedio = round(sum(compounds) / len(compounds), 4) if compounds else None
    etiqueta_consolidada = None
    if compound_promedio is not None:
        if compound_promedio > 0.05:
            etiqueta_consolidada = "BULLISH"
        elif compound_promedio < -0.05:
            etiqueta_consolidada = "BEARISH"
        else:
            etiqueta_consolidada = "NEUTRAL"

    return {
        "total"                    : len(docs),
        "filtro_aplicado"          : {"fuente": fuente, "sentimiento": sentimiento, "ticker": ticker},
        "compound_promedio"        : compound_promedio,
        "sentimiento_consolidado"  : etiqueta_consolidada,
        "mensaje"                  : None if docs else "No hay noticias que coincidan con el filtro seleccionado.",
        "noticias"                 : docs
    }

print("✓ Endpoint /api/noticias?fuente=&sentimiento=&ticker=&limite= registrado")


✓ Endpoint /api/noticias?fuente=&sentimiento=&ticker=&limite= registrado


## Paso 12 — Endpoint `/api/estrategias` *(Extra — Optimizador de Portafolio Markowitz)*

In [107]:
# ============================================================
# PASO 12 (EXTRA): Endpoint del optimizador de portafolio (Markowitz)
#
# Lee la coleccion 'estrategias' poblada por el Notebook 9. Cada
# documento es una combinacion perfil_riesgo x horizonte (3 x 4 = 12
# combinaciones), con la asignacion optima de los 5 tickers mineros.
#
# Alimenta la pestana "Estrategias" del frontend (controles Capital
# Inicial, Horizonte Temporal y Nivel de Riesgo). El parametro
# 'capital' escala 'asignacion_pct' a un monto en USD por activo,
# ya que MongoDB solo guarda porcentajes (no depende de un capital
# fijo).
# ============================================================

PERFILES_VALIDOS = {'conservador', 'moderado', 'agresivo'}
HORIZONTES_VALIDOS = {'3m', '6m', '1y', '3y'}

@app.get(
    "/api/estrategias",
    summary="Asignacion optima de portafolio (Markowitz) por perfil y horizonte",
    tags=["Estrategias"],
    response_description="Pesos optimos por activo, retorno/volatilidad esperados, Sharpe Ratio y frontera eficiente"
)
def estrategias(
    perfil_riesgo: str = Query(
        default="moderado",
        description="'conservador' (varianza minima), 'moderado' (max Sharpe) o 'agresivo' (max retorno)"
    ),
    horizonte: str = Query(
        default="1y",
        description="'3m', '6m', '1y' o '3y'"
    ),
    capital: float = Query(
        default=100000.0, gt=0,
        description="Capital inicial en USD; se usa para calcular 'monto_asignado_usd' por activo"
    )
):
    """
    Retorna la asignacion optima de portafolio calculada por el
    Notebook 9 (Teoria Moderna de Portafolio de Markowitz) para una
    combinacion de perfil de riesgo y horizonte temporal.

    **Fuente:** coleccion `estrategias` de MongoDB, poblada por el
    Notebook 9 (12 combinaciones: 3 perfiles x 4 horizontes).

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    p = perfil_riesgo.strip().lower()
    h = horizonte.strip().lower()

    if p not in PERFILES_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error": f"Perfil de riesgo '{perfil_riesgo}' no reconocido.",
                "perfiles_validos": sorted(PERFILES_VALIDOS)
            }
        )
    if h not in HORIZONTES_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error": f"Horizonte '{horizonte}' no reconocido.",
                "horizontes_validos": sorted(HORIZONTES_VALIDOS)
            }
        )

    proyeccion = {"_id": 0, "created_at": 0}
    doc = db[COL_ESTRATEGIAS].find_one({"perfil_riesgo": p, "horizonte": h}, proyeccion)

    if not doc:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay estrategia calculada para perfil='{p}', horizonte='{h}'.",
                "sugerencia" : "Ejecuta el Notebook 9 (Optimizador de Portafolio) para poblar la coleccion 'estrategias'.",
                "perfil_riesgo": p,
                "horizonte"    : h
            }
        )

    # Escalar la asignacion porcentual al capital inicial solicitado
    activos_escalados = []
    for activo in doc.get("activos", []):
        activo_escalado = dict(activo)
        activo_escalado["monto_asignado_usd"] = round(capital * activo.get("asignacion_pct", 0.0) / 100.0, 2)
        activos_escalados.append(activo_escalado)

    return {
        "perfil_riesgo"          : doc.get("perfil_riesgo"),
        "horizonte"              : doc.get("horizonte"),
        "capital_inicial_usd"    : capital,
        "retorno_esperado_anual" : doc.get("retorno_esperado_anual"),
        "volatilidad_anual"      : doc.get("volatilidad_anual"),
        "sharpe_ratio"           : doc.get("sharpe_ratio"),
        "optimizacion_exitosa"   : doc.get("optimizacion_exitosa"),
        "activos"                : activos_escalados,
        "frontera_eficiente"     : doc.get("frontera_eficiente", []),
        "fecha_calculo"          : doc.get("fecha_calculo"),
    }

print("\u2713 Endpoint /api/estrategias?perfil_riesgo=&horizonte=&capital= registrado")


✓ Endpoint /api/estrategias?perfil_riesgo=&horizonte=&capital= registrado


## Paso 13 — Endpoint `/api/backtests` *(Extra — Backtesting y Portafolio)*

In [108]:
# ============================================================
# PASO 13 (EXTRA): Endpoint de backtesting de portafolio
#
# Lee la coleccion 'backtests' poblada por el Notebook 10. Cada
# documento es una combinacion modelo x perfil_riesgo (5 modelos x
# 3 perfiles = 15 combinaciones), con la curva de equity, los trades
# y las metricas del backtest diversificado.
#
# Este UNICO endpoint alimenta DOS pestanas del frontend:
#   - "Backtesting": reporte tecnico completo (metricas, trades, curva).
#   - "Portafolio" : vista resumida (P&L, posiciones, distribucion).
#
# El parametro 'capital' reescala los montos en USD (equity_curve,
# posiciones, P&L) proporcionalmente al capital base con el que se
# corrio el backtest en el Notebook 10 (CAPITAL_BASE = 100,000 USD).
# Las metricas expresadas en porcentaje (retorno, Sharpe, Sortino,
# Max Drawdown, Win Rate) NO cambian al reescalar el capital.
# ============================================================

MODELOS_BACKTEST_VALIDOS = {'SVC', 'LSTM', 'BiLSTM', 'GRU', 'SimpleRNN'}

@app.get(
    "/api/backtests",
    summary="Backtest de portafolio diversificado (alimenta Backtesting y Portafolio)",
    tags=["Backtesting", "Portafolio"],
    response_description="Metricas del backtest, curva de equity, trades y posiciones finales por ticker"
)
def backtests(
    modelo: str = Query(
        default="SVC",
        description="'SVC', 'LSTM', 'BiLSTM', 'GRU' o 'SimpleRNN' — clasificador cuyas senales se simularon"
    ),
    perfil_riesgo: str = Query(
        default="moderado",
        description="'conservador', 'moderado' o 'agresivo' — pesos de asignacion usados (Notebook 9)"
    ),
    capital: float = Query(
        default=100000.0, gt=0,
        description="Capital inicial en USD; reescala la curva de equity y las posiciones proporcionalmente"
    )
):
    """
    Retorna el backtest de portafolio diversificado (5 tickers mineros,
    ponderados por Markowitz) para una combinacion de modelo de senales
    y perfil de riesgo.

    **Fuente:** coleccion `backtests` de MongoDB, poblada por el
    Notebook 10 (15 combinaciones: 5 modelos x 3 perfiles).

    Los campos `_id` y `created_at` se excluyen de la respuesta.
    """
    m = modelo.strip()
    p = perfil_riesgo.strip().lower()

    if m not in MODELOS_BACKTEST_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error": f"Modelo '{m}' no reconocido.",
                "modelos_validos": sorted(MODELOS_BACKTEST_VALIDOS)
            }
        )
    if p not in PERFILES_VALIDOS:
        raise HTTPException(
            status_code=400,
            detail={
                "error": f"Perfil de riesgo '{perfil_riesgo}' no reconocido.",
                "perfiles_validos": sorted(PERFILES_VALIDOS)
            }
        )

    proyeccion = {"_id": 0}
    doc = db[COL_BACKTESTS].find_one({"modelo": m, "perfil_riesgo": p}, proyeccion)

    if not doc:
        raise HTTPException(
            status_code=404,
            detail={
                "error"      : f"No hay backtest calculado para modelo='{m}', perfil='{p}'.",
                "sugerencia" : "Ejecuta el Notebook 10 (Backtesting de Portafolio) para poblar la coleccion 'backtests'.",
                "modelo"        : m,
                "perfil_riesgo" : p
            }
        )

    capital_base = doc.get("capital_base", 100000.0) or 100000.0
    escala = capital / capital_base

    equity_curve_escalada = [
        {"fecha": punto["fecha"], "valor": round(punto["valor"] * escala, 2)}
        for punto in doc.get("equity_curve", [])
    ]

    posiciones_escaladas = []
    for pos in doc.get("posiciones_finales", []):
        pos_escalada = dict(pos)
        for campo in ("capital_inicial_sleeve", "valor_actual", "pnl_usd"):
            if pos_escalada.get(campo) is not None:
                pos_escalada[campo] = round(pos_escalada[campo] * escala, 2)
        posiciones_escaladas.append(pos_escalada)

    trades_escalados = doc.get("trades", [])  # los trades usan retorno_pct, no dependen del capital

    return {
        "modelo"             : doc.get("modelo"),
        "perfil_riesgo"      : doc.get("perfil_riesgo"),
        "horizonte_pesos"    : doc.get("horizonte_pesos"),
        "capital_inicial_usd": capital,
        "comision_pct"       : doc.get("comision_pct"),
        "slippage_pct"       : doc.get("slippage_pct"),
        "fecha_inicio"       : doc.get("fecha_inicio"),
        "fecha_fin"          : doc.get("fecha_fin"),
        "metricas"           : doc.get("metricas", {}),
        "equity_curve"       : equity_curve_escalada,
        "trades"             : trades_escalados,
        "posiciones_finales" : posiciones_escaladas,
    }

print("\u2713 Endpoint /api/backtests?modelo=&perfil_riesgo=&capital= registrado (alimenta Backtesting y Portafolio)")


✓ Endpoint /api/backtests?modelo=&perfil_riesgo=&capital= registrado (alimenta Backtesting y Portafolio)


## Paso 14 — Endpoints `/api/ordenes` *(Broker: libro de órdenes manual)*

In [ ]:
# ============================================================
# PASO 14: Libro de órdenes manual del módulo Broker
#
# A diferencia de Estrategias/Portafolio/Backtesting (simulaciones
# AUTOMÁTICAS basadas en los modelos), este es un libro de órdenes
# MANUAL: el usuario decide comprar/vender "a mano" desde el frontend,
# usando el precio real más reciente de MongoDB (no un precio inventado
# en el cliente). Es una colección independiente ('ordenes_simuladas'),
# sin relación con 'predicciones' ni 'backtests' — es intencional,
# para no mezclar "lo que sugiere la IA" con "lo que decide el usuario".
#
# La cuenta (saldo disponible + posiciones abiertas) NO se guarda como un
# documento aparte: se recalcula en cada consulta a partir de TODO el
# historial de 'ordenes_simuladas', partiendo de un saldo inicial fijo de
# paper trading. Esto evita que el saldo se pueda desincronizar del
# historial real de órdenes (siempre son la misma fuente de verdad).
#
# POST /api/ordenes  → registra una orden nueva (precio se calcula en
#                       el servidor a partir del último precio real).
#                       Valida fondos suficientes (BUY) y posición
#                       suficiente (SELL) antes de ejecutar.
# GET  /api/ordenes   → lista el historial de órdenes (más recientes primero)
# GET  /api/cuenta     → poder adquisitivo actual + posiciones abiertas
# ============================================================

from pydantic import BaseModel, Field

COMISION_ORDEN_PCT = 0.001        # 0.10%, igual que el Notebook 10
SALDO_INICIAL       = 100_000.00  # Saldo inicial de la cuenta paper trading


def calcular_cuenta():
    """
    Reconstruye el estado de la cuenta paper trading (saldo disponible y
    posiciones netas por ticker) reproduciendo, en orden, el efecto de
    TODAS las órdenes registradas en 'ordenes_simuladas' sobre un saldo
    inicial fijo (SALDO_INICIAL).

    No se persiste como un documento de "cuenta" separado a propósito:
    al derivarse siempre del historial de órdenes, nunca puede quedar
    desincronizada de él.

    Retorna: (saldo_disponible: float, posiciones: dict[str, float])
    """
    saldo = SALDO_INICIAL
    posiciones = {}

    cursor = db[COL_ORDENES].find(
        {}, {"ticker": 1, "direccion": 1, "cantidad": 1, "total": 1}
    )
    for o in cursor:
        t = o["ticker"]
        cantidad = o["cantidad"]
        if o["direccion"] == "BUY":
            saldo -= o["total"]
            posiciones[t] = posiciones.get(t, 0) + cantidad
        else:  # SELL
            saldo += o["total"]
            posiciones[t] = posiciones.get(t, 0) - cantidad

    # Solo se muestran posiciones netas positivas (no se permiten posiciones
    # cortas en este simulador; ver validación de SELL en crear_orden).
    posiciones = {t: c for t, c in posiciones.items() if c > 0}
    return round(saldo, 2), posiciones


class NuevaOrden(BaseModel):
    ticker: str
    direccion: str = Field(..., description="'BUY' o 'SELL'")
    cantidad: float = Field(..., gt=0)
    tipo_orden: str = Field(default="MARKET", description="'MARKET' o 'LIMIT'")

@app.post(
    "/api/ordenes",
    summary="Registrar una orden manual (Broker)",
    tags=["Broker"],
    response_description="La orden registrada, con precio real de ejecución y comisión calculados en el servidor"
)
def crear_orden(orden: NuevaOrden):
    """
    Registra una orden manual del usuario en el libro de órdenes.

    El **precio de ejecución** se toma del último precio real disponible
    en `precios_ohlcv` (Notebook 1) — el cliente NUNCA decide el precio,
    solo el ticker, dirección y cantidad. Esto evita que el frontend
    pueda enviar precios arbitrarios.

    Antes de ejecutar, la orden se valida contra el estado actual de la
    cuenta (calculado con `calcular_cuenta()`):
    - **BUY**: se rechaza si el costo total supera el poder adquisitivo.
    - **SELL**: se rechaza si la cantidad supera la posición actual del
      ticker (no se permiten posiciones en corto).

    **Body esperado:**
    ```json
    { "ticker": "FSM", "direccion": "BUY", "cantidad": 10, "tipo_orden": "MARKET" }
    ```
    """
    t = validar_ticker(orden.ticker)

    d = orden.direccion.strip().upper()
    if d not in {"BUY", "SELL"}:
        raise HTTPException(status_code=400, detail={"error": "direccion debe ser 'BUY' o 'SELL'."})

    saldo_actual, posiciones_actuales = calcular_cuenta()

    # ── Validar posición suficiente antes de vender ──────────────────
    if d == "SELL":
        cantidad_disponible = posiciones_actuales.get(t, 0)
        if orden.cantidad > cantidad_disponible:
            raise HTTPException(
                status_code=400,
                detail={
                    "error": f"No puedes vender {orden.cantidad} de '{t}': tu posición actual es {cantidad_disponible}.",
                    "posicion_actual": cantidad_disponible
                }
            )

    # ── Precio real más reciente (mismo criterio que /api/mercado) ──
    ultimo_doc = db[COL_PRECIOS].find_one({"ticker": t}, sort=[("fecha", -1)])
    if not ultimo_doc:
        raise HTTPException(
            status_code=404,
            detail={"error": f"No hay precios disponibles para '{t}'.", "sugerencia": "Ejecuta el Notebook 1 (Ingesta)."}
        )

    precio_ejecucion = float(ultimo_doc["close"])
    subtotal = precio_ejecucion * orden.cantidad
    comision = round(subtotal * COMISION_ORDEN_PCT, 2)
    total = round(subtotal + comision, 2) if d == "BUY" else round(subtotal - comision, 2)

    # ── Validar fondos suficientes antes de comprar ──────────────────
    if d == "BUY" and total > saldo_actual:
        raise HTTPException(
            status_code=400,
            detail={
                "error": f"Fondos insuficientes: la orden cuesta ${total:.2f} y tu poder adquisitivo es ${saldo_actual:.2f}.",
                "poder_adquisitivo": saldo_actual
            }
        )

    documento = {
        "ticker"          : t,
        "direccion"       : d,
        "cantidad"        : orden.cantidad,
        "tipo_orden"      : orden.tipo_orden.strip().upper(),
        "precio_ejecucion": round(precio_ejecucion, 4),
        "subtotal"        : round(subtotal, 2),
        "comision"        : comision,
        "total"           : total,
        "fecha"           : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "created_at"      : datetime.now()
    }

    resultado = db[COL_ORDENES].insert_one(documento)
    documento["_id"] = str(resultado.inserted_id)

    return documento


@app.get(
    "/api/ordenes",
    summary="Historial de órdenes manuales (Broker)",
    tags=["Broker"],
    response_description="Lista de órdenes registradas, más recientes primero"
)
def listar_ordenes(ticker: Optional[str] = Query(default=None), limite: int = Query(default=50, le=200)):
    """
    Retorna el historial de órdenes manuales registradas desde el módulo Broker.

    **Parámetros opcionales:**
    - `ticker`: filtra por un ticker específico.
    - `limite`: máximo de órdenes a devolver (por defecto 50, máximo 200).
    """
    filtro = {}
    if ticker:
        filtro["ticker"] = ticker.strip()

    cursor = db[COL_ORDENES].find(filtro, {"created_at": 0}).sort("created_at", -1).limit(limite)
    ordenes = []
    for doc in cursor:
        doc["_id"] = str(doc["_id"])
        ordenes.append(doc)

    return {"total": len(ordenes), "ordenes": ordenes}


@app.get(
    "/api/cuenta",
    summary="Estado de la cuenta paper trading (Broker)",
    tags=["Broker"],
    response_description="Poder adquisitivo disponible y posiciones abiertas, derivados del historial de órdenes"
)
def obtener_cuenta():
    """
    Retorna el estado actual de la cuenta simulada del Broker, recalculado
    en cada consulta a partir de TODO el historial de 'ordenes_simuladas'
    (ver `calcular_cuenta()`), por lo que siempre está sincronizada.

    - `saldo_inicial`: capital con el que arranca la cuenta paper trading.
    - `poder_adquisitivo`: efectivo disponible ahora mismo.
    - `posiciones`: lista de posiciones abiertas, cada una con la cantidad
      neta y su valor de mercado según el precio real más reciente.
    """
    saldo, posiciones_cantidad = calcular_cuenta()

    posiciones = []
    for t, cantidad in posiciones_cantidad.items():
        doc = db[COL_PRECIOS].find_one({"ticker": t}, sort=[("fecha", -1)])
        precio_actual = float(doc["close"]) if doc else None
        posiciones.append({
            "ticker"       : t,
            "cantidad"     : cantidad,
            "precio_actual": round(precio_actual, 4) if precio_actual is not None else None,
            "valor_mercado": round(precio_actual * cantidad, 2) if precio_actual is not None else None
        })

    return {
        "saldo_inicial"    : SALDO_INICIAL,
        "poder_adquisitivo": saldo,
        "posiciones"       : posiciones
    }

print("✓ Endpoints POST /api/ordenes, GET /api/ordenes y GET /api/cuenta registrados")


## Paso 15 — Levantar servidor y exponer con ngrok

In [109]:
# ============================================================
# PASO 14: Levantar el servidor FastAPI y el túnel ngrok
#
# Secuencia correcta de arranque:
#   1. pkill -9 ngrok  → matar proceso a nivel OS
#   2. time.sleep(3)   → esperar que el proceso muera
#   3. ngrok.kill()    → limpiar estado interno de pyngrok
#   4. time.sleep(2)   → esperar que ngrok.io libere el túnel
#   5. ngrok.connect() → abrir túnel nuevo y limpio
#
# ⚠ No mover estas líneas ni cambiar el orden.
# ⚠ NO abrir un segundo túnel en ninguna otra celda.
# ============================================================

# ── Paso 10.1: limpiar procesos ngrok anteriores y liberar puerto 8000 ─────
!pkill -9 ngrok 2>/dev/null || true
time.sleep(3)

try:
    ngrok.kill()
except Exception:
    pass
time.sleep(2)

# Kill any process using port 8000 (uvicorn, etc.)
!lsof -t -i:8000 | xargs kill -9 2>/dev/null || true
time.sleep(2)

# ── Paso 10.2: abrir el túnel ngrok ──────────────────────────
tunel = ngrok.connect(8000)
URL_PUBLICA = tunel.public_url

print("=" * 68)
print(f"  URL PÚBLICA DE LA API : {URL_PUBLICA}")
print("=" * 68)
print()
print(f"  ✓  Salud   :  {URL_PUBLICA}/api/salud")
print(f"  ✓  Mercado :  {URL_PUBLICA}/api/mercado/BVN")
print(f"  ✓  SVC     :  {URL_PUBLICA}/api/svc/BVN")
print(f"  ✓  RNNs    :  {URL_PUBLICA}/api/rnns/BVN")
print(f"  ✓  LSTM    :  {URL_PUBLICA}/api/lstm/BVN")
print(f"  ✓  Noticias:  {URL_PUBLICA}/api/noticias?fuente=all&sentimiento=all")
print(f"  ✓  Estrategias: {URL_PUBLICA}/api/estrategias?perfil_riesgo=moderado&horizonte=1y")
print(f"  ✓  Backtests  : {URL_PUBLICA}/api/backtests?modelo=SVC&perfil_riesgo=moderado")
print(f"  ✓  Cuenta   :  {URL_PUBLICA}/api/cuenta")
print(f"  ✓  Swagger :  {URL_PUBLICA}/docs")
print()
print("  >>> COPIA LA URL PÚBLICA DE ARRIBA <<<")
print("  >>> Pégala en index.html del frontend (campo API_URL) <<<")
print()
print("  ⚠  Mantén este notebook corriendo mientras uses el frontend.")

# ── Paso 10.3: arrancar uvicorn en un hilo aparte ────────────
def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo al servidor para iniciar

print()
print("  ✓ Servidor FastAPI activo en el puerto 8000")
print("  ✓ API lista para recibir solicitudes del frontend")

  URL PÚBLICA DE LA API : https://regular-deduce-sanctity.ngrok-free.dev

  ✓  Salud   :  https://regular-deduce-sanctity.ngrok-free.dev/api/salud
  ✓  Mercado :  https://regular-deduce-sanctity.ngrok-free.dev/api/mercado/BVN
  ✓  SVC     :  https://regular-deduce-sanctity.ngrok-free.dev/api/svc/BVN
  ✓  RNNs    :  https://regular-deduce-sanctity.ngrok-free.dev/api/rnns/BVN
  ✓  LSTM    :  https://regular-deduce-sanctity.ngrok-free.dev/api/lstm/BVN
  ✓  Noticias:  https://regular-deduce-sanctity.ngrok-free.dev/api/noticias?fuente=all&sentimiento=all
  ✓  Estrategias: https://regular-deduce-sanctity.ngrok-free.dev/api/estrategias?perfil_riesgo=moderado&horizonte=1y
  ✓  Backtests  : https://regular-deduce-sanctity.ngrok-free.dev/api/backtests?modelo=SVC&perfil_riesgo=moderado
  ✓  Swagger :  https://regular-deduce-sanctity.ngrok-free.dev/docs

  >>> COPIA LA URL PÚBLICA DE ARRIBA <<<
  >>> Pégala en index.html del frontend (campo API_URL) <<<

  ⚠  Mantén este notebook corriendo mientra

## Paso 16 — Verificación final del sistema

In [110]:
# ============================================================
# PASO 15: Verificar que los endpoints principales responden
# correctamente antes de conectar el frontend.
#
# Ejecuta esta celda DESPUÉS del Paso 10.
# ============================================================

import requests

TICKERS_PRUEBA = ['FSM', 'BVN']
CABECERAS = {'ngrok-skip-browser-warning': '1'}  # Evita la pantalla de advertencia de ngrok

print("=" * 68)
print("VERIFICACIÓN FINAL — ENDPOINTS DE LA API")
print("=" * 68)

# ── 1. /api/salud ─────────────────────────────────────────────
print("\n1️⃣  /api/salud")
try:
    r = requests.get(f"{URL_PUBLICA}/api/salud", headers=CABECERAS, timeout=10)
    d = r.json()
    estado_bd = d.get('base_datos', '—')
    estado    = d.get('estado', '—')
    col_str   = ", ".join(f"{k}: {v}" for k,v in d.get('colecciones', {}).items())
    print(f"   Estado: {estado} | BD: {estado_bd}")
    print(f"   Colecciones — {col_str}")
    if d.get('advertencia'):
        print(f"   ⚠  {d['advertencia']}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 2. /api/mercado/{ticker} ──────────────────────────────────
print(f"\n2️⃣  /api/mercado/{{ticker}}")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/mercado/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            print(f"   ✓ {ticker:<12} — {d['registros']} registros | "
                  f"último precio: ${d.get('ultimo_precio', '—')} | "
                  f"rango: {d['fecha_inicio']} → {d['fecha_fin']}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 3. /api/svc/{ticker} ─────────────────────────────────────
print(f"\n3️⃣  /api/svc/{{ticker}}")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/svc/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            senal = d.get('prediccion', {}).get('senal', '—')
            acc   = d.get('metricas',   {}).get('accuracy', None)
            acc_s = f"{acc*100:.1f}%" if acc is not None else "—"
            print(f"   ✓ {ticker:<12} — Señal: {senal:<5} | Accuracy: {acc_s}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 4. /api/rnns/{ticker}?modelo=LSTM ─────────────────────────
print(f"\n4️⃣  /api/rnns/{{ticker}}?modelo=LSTM (Notebook 9)")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/rnns/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            senal = d.get('prediccion', {}).get('senal', '—')
            acc   = d.get('metricas',   {}).get('accuracy', None)
            acc_s = f"{acc*100:.1f}%" if acc is not None else "—"
            print(f"   ✓ {ticker:<12} — Señal: {senal:<5} | Accuracy: {acc_s}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 5. /api/lstm/{ticker} (Regresor) ──────────────────────────
print(f"\n5️⃣  /api/lstm/{{ticker}} (Notebook 10)")
for ticker in TICKERS_PRUEBA:
    try:
        r = requests.get(f"{URL_PUBLICA}/api/lstm/{ticker}", headers=CABECERAS, timeout=10)
        if r.status_code == 200:
            d = r.json()
            precio_actual = d.get('precio_actual_usd', '—')
            horizonte_60d = d.get('prediccion_futura', {}).get('60_dias', {}).get('precio', '—')
            rmse_usd      = d.get('metricas', {}).get('rmse_usd', None)
            rmse_s = f"RMSE: ${rmse_usd:.2f}" if rmse_usd is not None else "—"
            print(f"   ✓ {ticker:<12} — Precio actual: ${precio_actual} | Pred. 60d: ${horizonte_60d} | {rmse_s}")
        else:
            print(f"   ⚠ {ticker:<12} — HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
    except Exception as e:
        print(f"   ❌ {ticker:<12} — {e}")

# ── 6. /api/noticias ──────────────────────────────────────────
print(f"\n6️⃣  /api/noticias")
try:
    r = requests.get(f"{URL_PUBLICA}/api/noticias", params={"fuente": "all", "sentimiento": "all"},
                      headers=CABECERAS, timeout=10)
    if r.status_code == 200:
        d = r.json()
        print(f"   ✓ total={d.get('total','—')} | sentimiento consolidado: {d.get('sentimiento_consolidado','—')} "
              f"(compound={d.get('compound_promedio','—')})")
    else:
        print(f"   ⚠ HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 7. /api/estrategias (Notebook 9) ──────────────────────────
print(f"\n7️⃣  /api/estrategias")
try:
    r = requests.get(f"{URL_PUBLICA}/api/estrategias", params={"perfil_riesgo": "moderado", "horizonte": "1y"},
                      headers=CABECERAS, timeout=10)
    if r.status_code == 200:
        d = r.json()
        n_activos = len(d.get('activos', []))
        print(f"   ✓ perfil={d.get('perfil_riesgo','—')} | horizonte={d.get('horizonte','—')} | "
              f"retorno esp.: {d.get('retorno_esperado_anual','—')} | sharpe: {d.get('sharpe_ratio','—')} | "
              f"{n_activos} activos asignados")
    else:
        print(f"   ⚠ HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 8. /api/backtests (Notebook 10 — Backtesting + Portafolio) ──
print(f"\n8️⃣  /api/backtests")
try:
    r = requests.get(f"{URL_PUBLICA}/api/backtests", params={"modelo": "SVC", "perfil_riesgo": "moderado"},
                      headers=CABECERAS, timeout=10)
    if r.status_code == 200:
        d = r.json()
        m = d.get('metricas', {})
        print(f"   ✓ modelo={d.get('modelo','—')} | perfil={d.get('perfil_riesgo','—')} | "
              f"retorno: {m.get('total_return_pct','—')}% | sharpe: {m.get('sharpe_ratio','—')} | "
              f"trades: {m.get('total_trades','—')}")
    else:
        print(f"   ⚠ HTTP {r.status_code}: {r.json().get('detail', {}).get('error', '—')}")
except Exception as e:
    print(f"   ❌ Error: {e}")

# ── 9. Swagger UI ────────────────────────────────────────────
print(f"\n9️⃣  Documentación interactiva (Swagger UI)")
print(f"   🔗 {URL_PUBLICA}/docs")

print("\n" + "=" * 68)
print("  ✅ Verificación completa")
print(f"  🔗 Pega esta URL en index.html del frontend:")
print(f"     const API_URL = \"{URL_PUBLICA}\";")
print("=" * 68)


VERIFICACIÓN FINAL — ENDPOINTS DE LA API

1️⃣  /api/salud
   Estado: operativo | BD: conectada
   Colecciones — precios_ohlcv: 1253, predicciones: 29, metricas_modelos: 29, estrategias: 12, backtests: 12

2️⃣  /api/mercado/{ticker}
   ✓ FSM          — 251 registros | último precio: $8.52 | rango: 2025-07-11 → 2026-07-10
   ✓ BVN          — 251 registros | último precio: $30.0 | rango: 2025-07-11 → 2026-07-10

3️⃣  /api/svc/{ticker}
   ✓ FSM          — Señal: BUY   | Accuracy: 43.9%
   ✓ BVN          — Señal: SELL  | Accuracy: 61.0%

4️⃣  /api/rnns/{ticker}?modelo=LSTM (Notebook 9)
   ✓ FSM          — Señal: SELL  | Accuracy: 59.5%
   ✓ BVN          — Señal: SELL  | Accuracy: 62.2%

5️⃣  /api/lstm/{ticker} (Notebook 10)
   ✓ FSM          — Precio actual: $8.57 | Pred. 60d: $9.9031 | RMSE: $1.34
   ✓ BVN          — Precio actual: $29.55 | Pred. 60d: $33.9572 | RMSE: $3.90

6️⃣  /api/noticias
   ✓ total=46 | sentimiento consolidado: BULLISH (compound=0.1998)

7️⃣  /api/estrategias
   ✓ pe

## ⚠ Importante — Mientras el frontend esté en uso

- **Mantén este notebook corriendo** en Google Colab. Si se cierra o el runtime
  se desconecta, el túnel ngrok muere y el frontend mostrará "Error API".
- **La URL de ngrok cambia cada vez** que reinicias el kernel o el túnel.
  Cuando eso pase, copia la nueva URL del Paso 10 y actualiza la constante
  `API_URL` en `index.html`.
- Para la **exposición final**, considera el bono Streamlit: ofrece una URL
  permanente que no depende de que Colab esté corriendo.
